# importing libraries and initialize environment variables

In [ ]:
from neo4j import GraphDatabase
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import LLMChain
from langchain_classic.prompts import ChatPromptTemplate
import time
import os
from dotenv import load_dotenv

load_dotenv()

True

# Extracting the schema of the Graph Database

- The dataset has missing values so when running the query to get the properties of the first node ,relationship and the second node there will be different number of properties as some properties will be null for some nodes then the query will return them as different records but we want to classify it as the same schema so the below code mitigates this by checking if the same labels of first and second nodes and the same type of the relationship was seen before then just update the properies .<br/><br/>
- then convert the schema as a string to be used in the prompt later .

In [2]:
query  =  """
MATCH (n) - [r] -> (m)
RETURN DISTINCT 
labels(n) as labels_first_node ,
keys(properties(n)) as first_node_properties ,
type(r) as relation_type,
keys(properties(r)) as relation_properties ,
labels(m) as labels_second_node ,
keys(properties(m)) as second_node_properties
"""

connection = GraphDatabase.driver(os.getenv("CONNECTION_URL") 
              ,auth = (os.getenv("USER_NAME") 
                       ,os.getenv("PASSWORD"))
              )

In [3]:
# create new entry of the schema
def helper_function(labels_first_node ,first_node_properties
                    ,relation_type ,relation_properties
                    ,labels_second_node ,second_node_properties):
    
    temp = {"labels_first_node" : labels_first_node
                ,"first_node_properties" : set(first_node_properties)
                ,"relation_type" : relation_type
                ,"relation_properties" : set(relation_properties)
                ,"labels_second_node" : labels_second_node
                ,"second_node_properties" : set(second_node_properties)}
    return temp

schema = []
for record in connection.execute_query(query).records:

    labels_first_node = record["labels_first_node"]
    first_node_properties = record["first_node_properties"]
    relation_type = record["relation_type"]
    relation_properties = record["relation_properties"]
    labels_second_node = record["labels_second_node"]
    second_node_properties = record["second_node_properties"]

    # when the schema is not empty
    if schema :
        for entry in schema:
            if (entry["labels_first_node"] == labels_first_node and
                entry["labels_second_node"] ==  labels_second_node and 
                entry["relation_type"] == relation_type):

                # update first node properties
                first_node_properties = set(first_node_properties)
                entry["first_node_properties"].union(first_node_properties)
                # update econd node properties
                second_node_properties = set(second_node_properties)
                entry["second_node_properties"].union(second_node_properties)
                # update relation properties
                relation_properties = set(relation_properties)
                entry["relation_properties"].union(relation_properties)
                break

        # when the schema is not empty and we saw new entry
        else :
            temp = helper_function(labels_first_node ,first_node_properties
                            ,relation_type ,relation_properties
                            ,labels_second_node ,second_node_properties)
            schema.append(temp)
    # when the schema is empty           
    else:
        temp = helper_function(labels_first_node ,first_node_properties
                            ,relation_type ,relation_properties
                            ,labels_second_node ,second_node_properties)
        schema.append(temp)

In [4]:
temp = []
for entry in schema:
    labels_first_node =  ":".join(entry["labels_first_node"])
    labels_second_node =  ":".join(entry["labels_second_node"])
    relation_type = entry["relation_type"]
    first_node_properties =  ("{" 
                              + " ,".join(entry["first_node_properties"]) 
                              + "}")
    second_node_properties =  ("{" 
                              + " ,".join(entry["second_node_properties"]) 
                              + "}")
    relation_properties = ("{" 
                              + " ,".join(entry["relation_properties"]) 
                              + "}") 
    
    first_part = labels_first_node + first_node_properties
    second_part = labels_second_node + second_node_properties
    temp.append(f"{first_part} - [:{relation_type} {relation_properties}]"
          f" -> {second_part}")

schema = "\n".join(temp)

In [5]:
print(schema)

Person{poster ,name ,tmdbId ,imdbId ,born ,url ,bornIn ,bio} - [:ACTED_IN {role}] -> Movie{genres ,movieId ,birthYear ,tagline ,deathYear ,personType ,name ,personId ,releaseYear ,title ,avgVote}
Person{poster ,name ,tmdbId ,imdbId ,born ,url ,bornIn ,bio} - [:DIRECTED {}] -> Movie{genres ,movieId ,birthYear ,tagline ,deathYear ,personType ,name ,personId ,releaseYear ,title ,avgVote}


# Creating the first chain and the second chain

In [6]:
llm = ChatGoogleGenerativeAI(model = "models/gemini-2.5-flash"
                                    ,temperature = 0
                                    ,api_key = os.getenv("GEMANI_KEY"))

In [7]:
cypher_prompt = ChatPromptTemplate.from_template("""
You are an expert Neo4j Cypher query generator.

Your task is to convert a natural language question into a valid Cypher query.

STRICT RULES:
- Return ONLY the Cypher query.
- Do NOT include explanations.
- Do NOT include markdown or ``` blocks.
- Do NOT include comments.
- Use ONLY the provided schema.
- Do NOT invent labels, relationships, or properties.
- If the question cannot be answered using the schema, return: ERROR

SCHEMA:
{schema}

QUESTION:
{question}

CYPHER QUERY:
""").partial(schema=schema)

generate_cypher = LLMChain(llm = llm ,prompt = cypher_prompt)

/var/folders/lf/c33www711gv9_vm4d8ht63ww0000gn/T/ipykernel_62801/144852566.py:24: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  generate_cypher = LLMChain(llm = llm ,prompt = cypher_prompt)


In [8]:
response_prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant that answers questions using ONLY the provided database results.

STRICT RULES:
- Base your answer ONLY on the results.
- Do NOT add any external knowledge.
- If the results are empty, say: "No results found."
- Be concise and clear.
- If the results contain multiple items, format them as a list.

QUESTION:
{question}

RESULTS:
{results}

ANSWER:
""")
generate_answer = LLMChain(llm = llm ,prompt = response_prompt)

# Experimental Evaluation: NL Queries and Cypher Generation

- Questions :<br/>
 1 - which movies does Robin Wright acted in ?<br/>
 2 - what is the role that Johnny Depp played in Steven Spielberg ? <br/>
 3 - what are the movies that Quentin Tarantino directed ?<br/>
 4 - in what year was the movie Dancer in the Dark released ? <br/>
- you can try your own questions if you want .

In [ ]:
while True:

    question = input("write your question ot y to exsit : ")
    if question.lower() == "y":
        connection.close()
        break
    print("question : ")
    print(question ,"\n")

    query = generate_cypher.invoke({"question" : question})
    print("the generated query : ")
    print(query["text"] ,"\n")
    try:
        result = connection.execute_query(query["text"])
        records = [dict(record) for record in result.records]  
    except:
        print("try again")   
        records = []

    answer = generate_answer.invoke({"results" : result
                        ,"question" : question})["text"]
    print("the generated answer : ")
    print(answer ,"\n")
    print("-" * 100)

    # wait for a minute to ask again
    time.sleep(60)